In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = os.environ.get("LANGSMITH_TRACING", "True")
os.environ["LANGSMITH_API_KEY"] = os.environ.get("LANGSMITH_API_KEY", "")
os.environ["LANGSMITH_PROJECT"] = os.environ.get("LANGSMITH_PROJECT", "default")

for key_name in ["OPENROUTER_API_KEY", "LANGSMITH_API_KEY"]:
    if os.environ.get(key_name):
        print(key_name, "loaded")
    else:
        print(key_name, "MISSING")

print("Tracing:", os.environ.get("LANGSMITH_TRACING"))
print("Project:", os.environ.get("LANGSMITH_PROJECT"))

OPENROUTER_API_KEY loaded
LANGSMITH_API_KEY loaded
Tracing: true
Project: observability


In [3]:
from langsmith import Client

client = Client()

print("connected")

connected


In [2]:
import time
from langsmith import traceable

@traceable(run_type="retriever")
def search_docs(question):
    time.sleep(0.2)
    return ["refund_policy.md", "faq.md"]

@traceable(run_type="llm")
def fake_llm(question, docs):
    time.sleep(0.4)
    return "Based on" + str(len(docs)) + "documents, here is the answer."

@traceable
def pipline(question):
    docs = search_docs(question)
    answer = fake_llm(question, docs)
    return answer

print(pipline("What is the refund policy?"))

Based on2documents, here is the answer.


In [2]:
from langchain_openrouter import ChatOpenRouter

llm = ChatOpenRouter(model="inclusionai/ling-3.0-flash-fin:free", temperature=0)

reply = llm.invoke("reply with exactly: Langsmith is running")

print(reply.content)

Langsmith is running


# Sample RAG 

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

documents = [
    "Branch hours vary by location. Please visit our website or contact your nearest branch for specific hours of operation.",
    "You will need to provide a valid ID, proof of address, and your Social Security Number or equivalent identification number.",
    "You can check your account balance by logging into our online banking portal or mobile app. You can also check your balance at an ATM.",
    "Log into your online banking account or mobile app, select 'Transfer Funds,' choose the accounts, and enter the amount you wish to transfer.",
    "Yes, you can set up direct deposit by providing your employer with your account number and our bank's routing number.",
    "Report a lost or stolen debit card immediately by calling our customer service at 1-800-123-4567 or through our mobile app.",
]

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = InMemoryVectorStore.from_texts(
    documents,
    embedding=embeddings
)

retriever = vector_store.as_retriever(
    search_kwargs={"k": 2}
)

found = retriever.invoke(
    "how do I report a lost debit card?"
)

for doc in found:
    print("-", doc.page_content)

d:\Langsmith\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3050.06it/s]


- Report a lost or stolen debit card immediately by calling our customer service at 1-800-123-4567 or through our mobile app.
- You can check your account balance by logging into our online banking portal or mobile app. You can also check your balance at an ATM.


## Trace RAG 

In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langsmith import traceable

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a support assistant, Answer ONLY using context below. \n\n{context}"),
    ("human", "{question}"),
]
)


@traceable(name="rag_answer")
def rag_answer(question):

    docs = retriever.invoke(question)

    context = ""
    for doc in docs:
        context = context + " - " + doc.page_content + "\n"

    message = prompt.format_messages(context=context, question=question)
    reply = llm.invoke(message)

    return reply.content



In [10]:
print(rag_answer("How check balance? "))

You can check your balance in the following ways:

1. **Online Banking Portal** – Log into your account on our website to view your balance.
2. **Mobile App** – Log into our mobile app to check your balance on the go.
3. **ATM** – Visit any ATM to check your balance.


## Build RAG Agent using LangGraph

In [11]:
from langchain_core.tools import tool

@tool
def get_order_status(order_id: str) -> str:
    """Look up the delivery status of an order by its ID"""
    orders = {"A123": "Shipped, arriving Tuesday", "8456": "Processing"}
    if order_id in orders:
        return orders[order_id]
    return "Order not found"

@tool
def calculate_refund(price: float, days_since_purchese: int) -> str:
    """Calculated the refund amount for an order."""
    if days_since_purchese > 30:
        return "Not eligible: Purchased more than 30 dyas ago."
    return "Eligible for a full refund of" + str(price)

tools = [get_order_status, calculate_refund]

In [24]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

llm_with_tools = llm.bind_tools(tools)

def call_model(state):
    """ The brain: look at the conversation so far and decide what to do. """
    reply = llm_with_tools.invoke(state["messages"])
    return  {"messages": [reply]}

builder = StateGraph(MessagesState)

builder.add_node("model", call_model)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "model")
builder.add_conditional_edges("model", tools_condition)
builder.add_edge("tools", "model")

agent = builder.compile()

In [21]:
print(agent)
print(type(agent))

<class 'langgraph.graph.state.CompiledStateGraph'>


In [25]:
question = "How can I open a new bank account?"

final_state = agent.invoke({
    "messages": [("human", question)]
})

for message in final_state["messages"]:
    message.pretty_print()

================================ Human Message =================================

How can I open a new bank account?
================================== Ai Message ==================================

Opening a new bank account typically involves the following steps:

1. **Choose a bank** – Research different banks to compare fees, interest rates, minimum balance requirements, and branch/ATM availability.

2. **Select an account type** – Decide between a checking account, savings account, or both, based on your needs.

3. **Gather required documents** – Usually you'll need:
   - A valid government-issued photo ID (e.g., driver's license, passport)
   - Your Social Security Number (SSN) or Tax Identification Number (TIN)
   - Proof of address (e.g., utility bill, lease agreement)
   - An initial deposit (if required)

4. **Apply** – You can do this:
   - **In person** at a local branch
   - **Online** through the bank's website
   - **By phone** with some banks

5. **Complete the applicat